# Payments Reconciliation Intelligence — operational demo

**Question:** Which payment discrepancies need attention, where is estimated exposure concentrated, and which cases should operations investigate first?

This notebook generates synthetic feeds, runs truth-blind DuckDB reconciliation, and produces a management report. It demonstrates financial operations automation and SQL/data engineering. No machine learning is required.

**Controlled synthetic benchmark only.** Results are not production accuracy, realized loss, or recovered money. Start with `pip install -r requirements-demo.txt` from the repository root, then open this notebook. Run all cells in order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run from the repository root or notebooks directory")
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import HTML, display
from src.data_generator import GenerationConfig, write_dataset
from src.operations import run_operations
from src.reporting import write_reports

OUTPUT = ROOT / "artifacts" / "phase2_demo"
AS_OF = "2026-02-10T00:00:00Z"  # explicit, reproducible, after all generated settlements
SEED = 42


## 1. Create raw feeds

Generation is a separate simulation step. The generator also writes a truth file for later evaluation. The operational engine receives only the internal, gateway and bank feeds; it cannot read that file.


In [ ]:
paths = write_dataset(OUTPUT / "data", GenerationConfig(seed=SEED))
internal = pd.read_csv(paths["internal"], dtype=str)
gateway = pd.read_csv(paths["gateway"], dtype=str)
settlement = pd.read_csv(paths["settlement"], dtype=str)
display(pd.DataFrame({"source": ["internal", "gateway", "settlement"],
                      "rows": [len(internal), len(gateway), len(settlement)]}))
display(internal.head(3))


## 2. Reconcile using SQL

`sql/01_reconciliation.sql` aggregates duplicate feeds before matching, `02_exceptions.sql` detects discrepancies, and `03_analytics.sql` creates operational metrics and the queue. Monetary comparisons use DuckDB DECIMAL values.

This is an event-time snapshot at the cutoff. Missing settlements become overdue only **after** the configured three-calendar-day window. A settlement already received late remains historical timing exposure, not current outstanding money.


In [ ]:
result = run_operations(internal, gateway, settlement, as_of=AS_OF)
display(result.tables["summary"].T.rename(columns={0: "value"}))
display(result.tables["exception_type_metrics"])


## 3. Review concentration and daily operations

The overall exception rate uses distinct flagged transaction IDs divided by all observed transaction IDs, including orphans. An internal-ledger-only rate is reported separately. Provider identity is `unknown` in V1 inputs; no synthetic provider is invented.


In [ ]:
cols = ["dimension_value", "internal_payments", "exception_transactions",
        "exception_transaction_rate", "current_exposure_inr", "overdue_cash_inr", "exposure_share"]
display(result.tables["merchant_metrics"][cols].sort_values("current_exposure_inr", ascending=False).head(10))
display(result.tables["payment_method_metrics"][cols])
display(result.tables["daily_metrics"][cols].head(10))
display(result.tables["settlement_delay_metrics"])


## 4. Investigate the queue

The queue has one row per transaction, with all triggered rules attached. Severity dominates the score; current estimated exposure and age rank cases within severity. Overlapping monetary signals use the maximum estimate per transaction. This avoids double counting but can understate independent discrepancies.

This is a reproducible investigation snapshot, not a live ticketing system. `open_review` does not prove that a dispute is unresolved in a real business.


In [ ]:
display(result.tables["investigation_queue"].head(15))
display(result.tables["aging_metrics"])


## 5. Compare an earlier cutoff

Future bank events are excluded. Pending and overdue counts change with the cutoff, independently of the truth labels. Actual historical availability cannot be reconstructed without ingestion timestamps; these inputs provide event timestamps only.


In [ ]:
earlier = run_operations(internal, gateway, settlement, as_of="2026-01-20T00:00:00Z")
comparison = pd.concat([
    earlier.tables["summary"].assign(snapshot="2026-01-20"),
    result.tables["summary"].assign(snapshot="2026-02-10"),
], ignore_index=True)
display(comparison[["snapshot", "internal_payments", "pending_payments", "overdue_payments",
                    "current_exposure_inr", "historical_late_cash_inr"]])


## 6. Produce management reporting

The HTML report contains charts and operational tables; complete CSVs, JSON summary and a reproducibility manifest sit beside it. No remote assets or web service are required.


In [ ]:
import hashlib
source_hashes = {paths[k].name: hashlib.sha256(paths[k].read_bytes()).hexdigest()
                 for k in ("internal", "gateway", "settlement")}
report = write_reports(result, OUTPUT / "reports", source_hashes=source_hashes)
display(HTML(report.read_text(encoding="utf-8")))
print(f"Open report: {report}")


## 7. Evaluate V1 rule compatibility separately

Only now do we read truth. Evaluation is restricted to the original ten V1 categories. Phase 2 adds bank gross/net checks; those signals can overlap V1 fee defects and have no separate injected labels. They are tested with hand-built fixtures, not mislabeled as false positives against an incomplete truth set.


In [ ]:
from src.evaluation import evaluate_detections
truth = pd.read_csv(paths["truth"], dtype={"transaction_id": str})
core_types = ["missing_gateway", "missing_settlement", "amount_mismatch", "status_mismatch",
              "duplicate_gateway", "duplicate_settlement", "fee_mismatch", "settlement_delay",
              "orphan_gateway", "orphan_settlement"]
detected_core = result.tables["detected_exceptions"].query("exception_type in @core_types")
evaluation = evaluate_detections(truth, detected_core)
display(pd.Series(evaluation.summary, name="controlled_V1_rule_benchmark"))


## Limitations

INR only; exact transaction-ID matching; calendar-day settlement SLA; one intended payment per internal transaction. Duplicate source rows are ambiguous, not proof of duplicate money movement. Refunds, partial/split settlements, chargebacks, business calendars, fee-contract verification, ingestion history, persistent case ownership/resolution, and real feeds are outside this phase. Financial exposure is a transparent review estimate.

See `docs/phase2_operations.md` for definitions, formulas and reproducibility instructions.
